=============================================================================
PIPELINE 2 V10 — LLM Data Extraction
=============================================================================

CHANGES FROM V9:
  - Dual input mode: unfiltered reports OR filtered documents
  - 10-K now PDF-based (was HTML)
  - Updated metrics list aligned with Pipeline 1's ESG_METRICS
  - Summary Excel matches v8 format (6 sheets)
  - Model/prompt testing support with run labels
  - Cost calculation updated for current pricing


# Pipeline 2: LLM Extraction

## Purpose
Extract ESG metrics from pre-filtered documents using Claude API.

## Prerequisites
- Run Pipeline 1 (Document Filtering) first
- Filtered documents in `filtered_documents/` folder
- Anthropic API key configured



In [ ]:
# =============================================================================
# GOOGLE COLAB SETUP
# =============================================================================

from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Drive mounted")

Mounted at /content/drive
✅ Google Drive mounted


In [ ]:
# =============================================================================
# INSTALL DEPENDENCIES
# =============================================================================

!pip install anthropic pymupdf openpyxl -q

print("✅ Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.6/832.6 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 54.8 MB/s eta 0:00:00
✅ Dependencies installed


In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================

import os
import re
import json
import time
import base64
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from datetime import datetime
from collections import Counter

import pandas as pd
import numpy as np
import fitz  # PyMuPDF
import anthropic

print("✅ Imports complete")

✅ Imports complete


## File Paths

In [ ]:
# =============================================================================
# PATH CONFIGURATION — UPDATED FOR DUAL INPUT MODE
# =============================================================================

DRIVE_BASE = Path("/content/drive/MyDrive")

# ── Input Mode Selection ──
# Set USE_FILTERED = False to run on original unfiltered reports (baseline test)
# Set USE_FILTERED = True and specify FILTER_VERSION to run on filtered docs
USE_FILTERED = True
FILTER_VERSION = "4"  # Which filtered_documents_X folder to use; 4 resulted in the best recall/precision

if USE_FILTERED:
    INPUT_DIR = DRIVE_BASE / "ESG Data" / f"filtered_documents_{FILTER_VERSION}"
    RUN_LABEL = f"filtered_v{FILTER_VERSION}"
else:
    INPUT_DIR = DRIVE_BASE / "ESG Data" / "reports"
    RUN_LABEL = "unfiltered"

# Output: tagged with run label and model
OUTPUT_DIR = DRIVE_BASE / "ESG Data" / "extracted_data" / RUN_LABEL
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Extraction logs
LOG_DIR = DRIVE_BASE / "ESG Data" / "extraction_logs" / RUN_LABEL
LOG_DIR.mkdir(parents=True, exist_ok=True)

YEARS = [2020, 2021, 2022, 2023, 2024]
SECTOR = "IT"

print(f"✅ Paths configured")
print(f"   Mode: {'FILTERED (v' + FILTER_VERSION + ')' if USE_FILTERED else 'UNFILTERED (full reports)'}")
print(f"   Run label: {RUN_LABEL}")
print(f"   Input: {INPUT_DIR}")
print(f"   Output: {OUTPUT_DIR}")


✅ Paths configured
   Mode: FILTERED (v4)
   Run label: filtered_v4
   Input: /content/drive/MyDrive/ESG Data/filtered_documents_4
   Output: /content/drive/MyDrive/ESG Data/extracted_data/filtered_v4


In [ ]:
# =============================================================================
# COMPANY LIST
# =============================================================================

TEST_COMPANIES = [
    # "Advanced_Micro_Devices",
    # "Nvidia",
    "Dell_Technologies",
    "HP_Inc",

]

# Full company list (uncomment when ready)
# ALL_COMPANIES = [
#     "Accenture", "Adobe_Inc", "Advanced_Micro_Devices", "Akamai_Technologies",
#     "Amphenol", "Analog_Devices", "Apple_Inc", "Applied_Materials", "Arista_Networks",
#     "Autodesk", "Broadcom", "Cadence_Design_Systems", "CDW_Corporation", "Cisco",
#     "Cognizant", "Corning_Inc", "CrowdStrike", "Datadog", "Dell_Technologies",
#     "Enphase_Energy", "EPAM_Systems", "F5_Inc", "Fair_Isaac", "First_Solar",
#     "Fortinet", "Gartner", "Gen_Digital", "GoDaddy", "Hewlett_Packard_Enterprise",
#     "HP_Inc", "IBM", "Intel", "Intuit", "Jabil", "Keysight_Technologies",
#     "KLA_Corporation", "Lam_Research", "Microchip_Technology", "Micron_Technology",
#     "Microsoft", "Monolithic_Power_Systems", "Motorola_Solutions", "NetApp", "Nvidia",
#     "NXP_Semiconductors", "ON_Semiconductor", "Oracle_Corporation", "Palantir_Technologies",
#     "Palo_Alto_Networks", "PTC_Inc", "Qualcomm", "Roper_Technologies", "Salesforce",
#     "Seagate_Technology", "ServiceNow", "Skyworks_Solutions", "Supermicro", "Synopsys",
#     "TE_Connectivity", "Teledyne_Technologies", "Teradyne", "Texas_Instruments",
#     "Trimble_Inc", "Tyler_Technologies", "Verisign", "Western_Digital", "Workday_Inc",
#     "Zebra_Technologies"
# ]


#print(f"✅ {len(ALL_COMPANIES)} companies in full list")
print(f"   Testing with: {TEST_COMPANIES}")

   Testing with: ['Dell_Technologies', 'HP_Inc']


## Extraction Parameters

In [ ]:
# =============================================================================
# EXTRACTION PARAMETERS — UPDATED WITH MODEL TESTING
# =============================================================================

from google.colab import userdata

# Model selection — change this to test different models
API_MODEL = "claude-haiku-4-5-20251001"

# Pricing per model ($/MTok) — update if prices change
MODEL_PRICING = {
    "claude-haiku-4-5-20251001": {"input": 1.00, "output": 5.00},
    "claude-sonnet-4-20250514": {"input": 3.00, "output": 15.00},
}

RATE_LIMITS = {
    "claude-sonnet-4-20250514": 400000,
    "claude-haiku-4-5-20251001": 400000,
}

EXTRACTION_CONFIG = {
    "max_tokens_output": 24000,
    "rate_limit_tokens_per_min": RATE_LIMITS.get(API_MODEL, 400000),
    "retry_on_rate_limit": True,
    "retry_wait_seconds": 60,
    "use_streaming": True,
}

print("✅ Extraction config loaded")
print(f"   Model: {API_MODEL}")
print(f"   Pricing: ${MODEL_PRICING.get(API_MODEL, {}).get('input', '?')}/MTok in, ${MODEL_PRICING.get(API_MODEL, {}).get('output', '?')}/MTok out")

# API key
try:
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    print("✅ API key loaded from Colab secrets")
except:
    ANTHROPIC_API_KEY = "your-api-key-here"
    print("⚠️ Using hardcoded API key - set in Colab secrets for security")

client = None
if ANTHROPIC_API_KEY and ANTHROPIC_API_KEY != "your-api-key-here":
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    print(f"✅ Anthropic client ready")
else:
    print("❌ Set ANTHROPIC_API_KEY in Colab secrets")

✅ Extraction config loaded
   Model: claude-haiku-4-5-20251001
   Pricing: $1.0/MTok in, $5.0/MTok out
✅ API key loaded from Colab secrets
✅ Anthropic client ready


## Prompts

### Sustainability Prompt

In [ ]:
PROMPT_SUSTAINABILITY = """Extract ESG metrics from this sustainability report for {company}.

METRICS TO EXTRACT ({metric_count} total):
{metrics}

═══════════════════════════════════════════════════════════════════════════════
STRICT RULES — VIOLATIONS WILL INVALIDATE THE EXTRACTION
═══════════════════════════════════════════════════════════════════════════════

SOURCE PRIORITY:
1. Data tables with exact numbers are the ONLY preferred source
2. Explicit numbers in text are acceptable ONLY if no table exists for that metric
3. NEVER extract values from charts, graphs, bar charts, or infographics unless they are clearly labeled
4. SCAN THE ENTIRE DOCUMENT before extracting — data tables are often at the END
5. IGNORE goals, targets, prjections or %-reductions comparing to previous years
6. IGNORE single country spotlights and country specific metric data for total or absolute values

YEAR RESTRICTION:
- ONLY years 2020, 2021, 2022, 2023, 2024
- IGNORE years outside of this range.

METRIC ID RESTRICTION:
- ONLY use metric_id values from the list above
- Do NOT invent new metric names

VALUE ACCURACY:
- Extract the EXACT value as shown: "1,234,567" → 1234567
- Do NOT round, estimate, or approximate
- Do NOT extract values that are only visible as bar heights or chart positions

UNIT EXTRACTION — CRITICAL:
- Extract the EXACT unit as shown in the document, not the expected unit
- If the document says "GWh", report "GWh" even if expected unit is "MWh"
- If the document says "GJ", report "GJ" even if expected unit is "MWh"
- If the document says "per 200,000 hours", report "per 200,000 hours"
- If the document says "megalitres", report "megalitres" not "m³"
- We will handle unit conversion later — your job is to report what the document shows

METRIC PRECISION — READ CAREFULLY:
- "Renewable Energy %" means ALL energy (electricity + fuel + heat), not just electricity
- "Renewable Electricity %" is a DIFFERENT metric — do not confuse them
- "Women in Workforce" means all employees, not just management or leadership
- "Women in Management" means management level only, not all employees
- "Women in Senior Management" means senior/executive management only
- "Total Waste" means all waste combined, not just one waste stream
- "Water Withdrawal" means total water intake, not consumption or discharge
- "Scope 2 Market-Based" and "Scope 2 Location-Based" are DIFFERENT — extract both separately
- If a metric says "Total" (e.g., Total Energy), extract the TOTAL row, not sub-categories
- When counting from tables (e.g., board members), count ALL rows, not just one category

CONFIDENCE:
- "high": Exact number from a data table
- "medium": Number from text or requires simple calculation
- "low": Estimated or ambiguous — PREFER TO SKIP rather than guess

CONTEXT:
- MUST include the table name, section header, or row label where the value was found
- Example: "Environmental Performance Data table, Scope 1 row, FY2023 column"

OUTPUT — Return ONLY a JSON array:
[
  {{"metric_id": "scope_1_emissions", "value": 6568, "unit": "tCO2e", "year": 2023, "confidence": "high", "context": "GHG Emissions table, Direct emissions row"}},
  {{"metric_id": "energy_total", "value": 1.2, "unit": "GWh", "year": 2023, "confidence": "high", "context": "Energy Performance table, Total energy row"}}
]

If no relevant data found, return: []
"""

print("✅ Sustainability Prompt V10.1 defined (strict extraction)")

✅ Sustainability Prompt V10.1 defined (strict extraction)


### DEF 14A Prompt

In [ ]:
PROMPT_DEF14A = """Extract governance data from this DEF 14A proxy filing for {company}.
This document covers FISCAL YEAR {fiscal_year}.

═══════════════════════════════════════════════════════════════════════════════
RULES:
═══════════════════════════════════════════════════════════════════════════════
1. Extract values that are EXPLICITLY STATED
2. SIMPLE CALCULATIONS ARE ALLOWED when data is clear:
   - "8 of 9 directors are independent" → board_independent = 8, board_size = 9
   - "all directors except the CEO are independent" + 9 total → 8 independent
   - Counting checkmarks in a diversity matrix table
   - SUM all female counts across age groups in the diversity matrix
3. ALL extracted values must have year: {fiscal_year}
4. Preserve exact decimals: "66.7%" → 66.7
5. For Y/N metrics: 1 = Yes/Exists, 0 = No/Not mentioned
6. ONLY use metric_id values listed below. Do NOT invent new names.
7. SCAN THE ENTIRE DOCUMENT before extracting — vote results are often in the Compensation Discussion section, not at the proposal.
8. Do NOT extract values from charts or graphs.
9. EVERY entry MUST include "context" describing where the value was found.
10. IGNORE goals, targets, prjections or %-reductions comparing to previous years.
11. IGNORE single country spotlights and country specific metric data for total or absolute values

═══════════════════════════════════════════════════════════════════════════════
METRICS TO EXTRACT:
═══════════════════════════════════════════════════════════════════════════════

BOARD COMPOSITION:

board_size (count)
- Count total directors listed or stated
- Look for "Board of Directors" section, director nominees table
- Typically 8-12 people
- EXCLUDE executive officers unless they are also directors

board_independent (count)
- Count directors marked as "independent"
- Look for independence table, checkmarks, or "X of Y directors are independent"
- Directors with "Non-Independent" mark are NOT independent
- The CEO is typically NOT independent

board_female (count)
- Count from Board Diversity Matrix (Female row)
- Count checkmarks or look for "X female directors"
- May be listed by age group — SUM ALL female counts across all categories
- Alternative: count directors with "Ms." or identified as female in bios
- If non-gender specific prefixes are used like "Dr." or "Prof." look further to see if might be female.

board_meetings (%)
- Board meeting attendance percentage
- Pattern: "all directors attended at least X%" → value = X (NOT 100)
- Pattern: "average attendance was X%" → value = X
- Examples:
  * "All directors attended at least 75% of meetings" → value = 75
  * "All directors attended at least 90 percent" → value = 90
  * "Average board attendance was 95%" → value = 95
  * "Each director attended 100% of meetings" → value = 100
- Only use 100 if explicitly stated as "100%" or "all meetings"

COMPENSATION:

ceo_pay_ratio (ratio)
- Look for "CEO Pay Ratio" section
- Format: "X to 1" or "X:1" → extract X as value
- Example: "216 to 1" → value = 216

say_on_pay (%)
- Look for "Say on Pay" or "Advisory Vote" RESULTS
- Extract the APPROVAL/FOR percentage, not the proposal description
- Look for: "received X% support", "X% of votes cast in favor", "X% approval"
- May appear in Compensation Discussion section, not just at the proposal
- Typical range: 70-99%

POLICIES (Yes=1, No=0):

ethics_policy (Y/N) — 1 if mentions: "Code of Ethics", "Code of Conduct", "Business Ethics Policy", "Standards of Business Conduct"
anti_corruption_policy (Y/N) — 1 if mentions: "Anti-Corruption Policy", "Anti-Bribery Policy", "FCPA Compliance"
whistleblower_policy (Y/N) — 1 if mentions: "Whistleblower Policy", "Whistleblower Hotline", "Ethics Hotline"
ethics_violations (count) — Count of violations mentioned, or 0 if "no violations" or not mentioned
dual_class_shares (Y/N) — 1 if dual-class structure (Class A/B with different voting rights), 0 if single class or not mentioned

═══════════════════════════════════════════════════════════════════════════════
OUTPUT — Return ONLY a JSON array:
═══════════════════════════════════════════════════════════════════════════════
[
  {{"metric_id": "board_size", "value": 9, "unit": "count", "year": {fiscal_year}, "confidence": "high", "context": "Nine directors listed in Director Nominees section"}},
  {{"metric_id": "board_independent", "value": 8, "unit": "count", "year": {fiscal_year}, "confidence": "high", "context": "8 of 9 directors are independent per Independence table"}},
  {{"metric_id": "board_female", "value": 3, "unit": "count", "year": {fiscal_year}, "confidence": "high", "context": "Diversity Matrix: 1 under 30 + 1 age 30-50 + 1 over 50 = 3 female"}},
  {{"metric_id": "ceo_pay_ratio", "value": 216, "unit": "ratio", "year": {fiscal_year}, "confidence": "high", "context": "CEO Pay Ratio section: 216 to 1"}},
  {{"metric_id": "say_on_pay", "value": 94.7, "unit": "%", "year": {fiscal_year}, "confidence": "high", "context": "Compensation Discussion: received 94.7% support at annual meeting"}}
]

If no relevant data found, return: []
"""

print("✅ DEF 14A Prompt V10.1 defined (strict extraction)")

✅ DEF 14A Prompt V10.1 defined (strict extraction)


### 10 K Prompt

In [ ]:
PROMPT_10K = """Extract tax and workforce data from this 10-K filing for {company}.
This document covers FISCAL YEAR {fiscal_year}.

═══════════════════════════════════════════════════════════════════════════════
RULES:
═══════════════════════════════════════════════════════════════════════════════
1. ALL extracted values must have year: {fiscal_year}
2. ONLY use metric_id values listed below
3. SIMPLE CALCULATIONS ARE ALLOWED:
   - Effective Tax Rate = (Provision for income taxes / Income before income taxes) × 100
   - If the rate is not stated directly but components are shown, calculate it
4. Extract the EXACT unit as shown in the document
5. EVERY entry MUST include "context" describing where the value was found
6. SCAN THE ENTIRE DOCUMENT — tax data is usually in the Notes to Financial Statements

═══════════════════════════════════════════════════════════════════════════════
METRICS TO EXTRACT:
═══════════════════════════════════════════════════════════════════════════════

effective_tax_rate (%)
- Look in Notes to Financial Statements, "Income Taxes" note
- May be in tax rate reconciliation table
- May be stated as "effective tax rate was X%"
- Can be NEGATIVE if net tax benefit instead of provision
- If not stated directly, calculate: (Provision for income taxes / Income before income taxes) × 100

accounting_controversies (count)
- Count of restatements or material weaknesses in internal controls
- 0 if none disclosed
- 1 if any material weakness, restatement, or SEC investigation mentioned

country_by_country_reporting (Y/N)
- 1 if geographic revenue breakdown by country/region is disclosed ANYWHERE in the filing
- 0 if no geographic revenue breakdown exists
- Look for: "Geographic Information", "Revenue by Region", "Segment Information" with country data

employees_total (count)
- Total employee headcount
- Usually in Item 1 (Business Description) or Human Capital section
- Look for: "approximately X employees", "X full-time employees"

═══════════════════════════════════════════════════════════════════════════════
OUTPUT — Return ONLY a JSON array:
═══════════════════════════════════════════════════════════════════════════════
[
  {{"metric_id": "effective_tax_rate", "value": 12.5, "unit": "%", "year": {fiscal_year}, "confidence": "high", "context": "Note 14 Income Taxes, rate reconciliation table"}},
  {{"metric_id": "employees_total", "value": 26196, "unit": "count", "year": {fiscal_year}, "confidence": "high", "context": "Item 1: approximately 26,196 employees as of fiscal year end"}}
]

If no relevant data found, return: []
"""


print("✅ 10-K Prompt V10.1 defined (strict extraction)")

✅ 10-K Prompt V10.1 defined (strict extraction)


## Rate Limiter

In [ ]:
# =============================================================================
# RATE LIMITER
# =============================================================================

class RateLimiter:
    """Rate limiter for API calls."""

    def __init__(self, tokens_per_minute: int = 400000):
        self.limit = tokens_per_minute
        self.tokens_used = 0
        self.window_start = time.time()

    def wait_if_needed(self, estimated_tokens: int):
        elapsed = time.time() - self.window_start

        if elapsed >= 60:
            self.tokens_used = 0
            self.window_start = time.time()
            elapsed = 0

        if self.tokens_used + estimated_tokens > self.limit:
            wait_time = 60 - elapsed + 2
            print(f"      ⏳ Rate limit: waiting {wait_time:.0f}s...")
            time.sleep(wait_time)
            self.tokens_used = 0
            self.window_start = time.time()

    def add_tokens(self, tokens: int):
        self.tokens_used += tokens

rate_limiter = RateLimiter(EXTRACTION_CONFIG['rate_limit_tokens_per_min'])
print(f"✅ Rate limiter: {rate_limiter.limit:,} tokens/min")

✅ Rate limiter: 400,000 tokens/min


## ESG Metrics List

In [ ]:
# =============================================================================
# ESG METRICS DEFINITION
# =============================================================================

ESG_METRICS = {
    # ═══════════════════════════════════════════════════════════════════════
    # ENVIRONMENT - Climate Change
    # ═══════════════════════════════════════════════════════════════════════
    "scope_1_emissions": {"pillar": "Environment", "theme": "Climate Change", "issue": "Carbon Emissions", "name": "Scope 1 GHG Emissions", "unit": "tCO2e", "type": "performance"},
    "scope_2_emissions_market": {"pillar": "Environment", "theme": "Climate Change", "issue": "Carbon Emissions", "name": "Scope 2 Market-Based", "unit": "tCO2e", "type": "performance"},
    "scope_2_emissions_location": {"pillar": "Environment", "theme": "Climate Change", "issue": "Carbon Emissions", "name": "Scope 2 Location-Based", "unit": "tCO2e", "type": "performance"},
    "scope_3_emissions": {"pillar": "Environment", "theme": "Climate Change", "issue": "Carbon Emissions", "name": "Scope 3 Emissions", "unit": "tCO2e", "type": "performance"},
    "carbon_intensity": {"pillar": "Environment", "theme": "Climate Change", "issue": "Carbon Emissions", "name": "GHG Intensity", "unit": "tCO2e/M USD", "type": "performance", "derived_from": ["scope_1_emissions", "scope_2_emissions_market", "revenue"]},
    "climate_targets": {"pillar": "Environment", "theme": "Climate Change", "issue": "Carbon Emissions", "name": "Science-Based Targets / Net Zero", "unit": "Y/N", "type": "policy"},

    # ENVIRONMENT - Energy
    "energy_total": {"pillar": "Environment", "theme": "Climate Change", "issue": "Energy", "name": "Total Energy Consumption", "unit": "MWh", "type": "performance", "component": True},
    "energy_renewable": {"pillar": "Environment", "theme": "Climate Change", "issue": "Energy", "name": "Renewable Energy Consumption", "unit": "MWh", "type": "performance", "component": True},
    "renewable_energy_pct": {"pillar": "Environment", "theme": "Climate Change", "issue": "Energy", "name": "Renewable Energy Percentage", "unit": "%", "type": "performance", "derived_from": ["energy_renewable", "energy_total"]},

    # ENVIRONMENT - Natural Capital
    "water_withdrawal": {"pillar": "Environment", "theme": "Natural Capital", "issue": "Water", "name": "Water Withdrawal", "unit": "m³", "type": "performance"},
    "water_recycled": {"pillar": "Environment", "theme": "Natural Capital", "issue": "Water", "name": "Water Recycled/Reused", "unit": "m³", "type": "performance"},

    # ENVIRONMENT - Pollution & Waste
    "waste_total": {"pillar": "Environment", "theme": "Pollution & Waste", "issue": "Waste", "name": "Total Waste Generated", "unit": "tonnes", "type": "performance", "component": True},
    "waste_recycled": {"pillar": "Environment", "theme": "Pollution & Waste", "issue": "Waste", "name": "Waste Recycled/Diverted", "unit": "tonnes", "type": "performance", "component": True},
    "hazardous_waste": {"pillar": "Environment", "theme": "Pollution & Waste", "issue": "Toxic Waste", "name": "Hazardous Waste", "unit": "tonnes", "type": "performance"},
    "ewaste_generated": {"pillar": "Environment", "theme": "Pollution & Waste", "issue": "E-Waste", "name": "E-Waste Generated", "unit": "tonnes", "type": "performance"},
    "ewaste_recycled": {"pillar": "Environment", "theme": "Pollution & Waste", "issue": "E-Waste", "name": "E-Waste Recycling Rate", "unit": "%", "type": "performance"},

    # ENVIRONMENT - Green Building
    "green_building_certifications": {"pillar": "Environment", "theme": "Environmental Opportunities", "issue": "Green Building", "name": "Green Building Certifications", "unit": "count", "type": "performance"},

    # ═══════════════════════════════════════════════════════════════════════
    # SOCIAL - Human Capital
    # ═══════════════════════════════════════════════════════════════════════

    # Workforce
    "employees_total": {"pillar": "Social", "theme": "Human Capital", "issue": "Labor", "name": "Total Employees", "unit": "count", "type": "performance", "component": True},
    "employee_turnover": {"pillar": "Social", "theme": "Human Capital", "issue": "Labor", "name": "Employee Turnover Rate", "unit": "%", "type": "performance"},

    # Health & Safety
    "safety_hours_worked": {"pillar": "Social", "theme": "Human Capital", "issue": "Health & Safety", "name": "Total Hours Worked", "unit": "hours", "type": "performance", "component": True},
    "lost_time_injuries": {"pillar": "Social", "theme": "Human Capital", "issue": "Health & Safety", "name": "Lost Time Injuries", "unit": "count", "type": "performance", "component": True},
    "recordable_injuries": {"pillar": "Social", "theme": "Human Capital", "issue": "Health & Safety", "name": "Recordable Injuries", "unit": "count", "type": "performance", "component": True},
    "ltifr": {"pillar": "Social", "theme": "Human Capital", "issue": "Health & Safety", "name": "Lost Time Injury Frequency Rate", "unit": "per M hours", "type": "performance", "derived_from": ["lost_time_injuries", "safety_hours_worked"]},
    "trir": {"pillar": "Social", "theme": "Human Capital", "issue": "Health & Safety", "name": "Total Recordable Injury Rate", "unit": "per M hours", "type": "performance", "derived_from": ["recordable_injuries", "safety_hours_worked"]},
    "fatalities": {"pillar": "Social", "theme": "Human Capital", "issue": "Health & Safety", "name": "Workplace Fatalities", "unit": "count", "type": "performance"},

    # Development
    "training_hours_total": {"pillar": "Social", "theme": "Human Capital", "issue": "Development", "name": "Total Training Hours", "unit": "hours", "type": "performance", "component": True},
    "training_hours_per_emp": {"pillar": "Social", "theme": "Human Capital", "issue": "Development", "name": "Training Hours per Employee", "unit": "hours/emp", "type": "performance", "derived_from": ["training_hours_total", "employees_total"]},

    # Diversity
    "women_workforce": {"pillar": "Social", "theme": "Human Capital", "issue": "Diversity", "name": "Women in Workforce", "unit": "%", "type": "performance"},
    "women_management": {"pillar": "Social", "theme": "Human Capital", "issue": "Diversity", "name": "Women in Management", "unit": "%", "type": "performance"},
    "women_senior_mgmt": {"pillar": "Social", "theme": "Human Capital", "issue": "Diversity", "name": "Women in Senior Management", "unit": "%", "type": "performance"},

    # Supply Chain
    "supplier_audits": {"pillar": "Social", "theme": "Human Capital", "issue": "Supply Chain", "name": "Supplier Audits Conducted", "unit": "count", "type": "performance"},
    "supplier_code_of_conduct": {"pillar": "Social", "theme": "Human Capital", "issue": "Supply Chain", "name": "Supplier Code of Conduct", "unit": "Y/N", "type": "policy"},

    # ═══════════════════════════════════════════════════════════════════════
    # SOCIAL - Product Liability
    # ═══════════════════════════════════════════════════════════════════════
    "data_breaches": {"pillar": "Social", "theme": "Product Liability", "issue": "Privacy", "name": "Data Breaches", "unit": "count", "type": "performance"},

    # ═══════════════════════════════════════════════════════════════════════
    # SOCIAL - Community
    # ═══════════════════════════════════════════════════════════════════════
    "community_investment": {"pillar": "Social", "theme": "Stakeholder Opposition", "issue": "Community", "name": "Community Investment", "unit": "USD", "type": "performance"},

    # ═══════════════════════════════════════════════════════════════════════
    # GOVERNANCE - Corporate Governance (Board) - Source: DEF 14A
    # ═══════════════════════════════════════════════════════════════════════
    "board_size": {"pillar": "Governance", "theme": "Corporate Governance", "issue": "Board", "name": "Board Size", "unit": "count", "type": "performance", "source": "DEF14A", "component": True},
    "board_independent": {"pillar": "Governance", "theme": "Corporate Governance", "issue": "Board", "name": "Independent Directors", "unit": "count", "type": "performance", "source": "DEF14A", "component": True},
    "board_independence_pct": {"pillar": "Governance", "theme": "Corporate Governance", "issue": "Board", "name": "Board Independence", "unit": "%", "type": "performance", "source": "DEF14A", "derived_from": ["board_independent", "board_size"]},
    "board_female": {"pillar": "Governance", "theme": "Corporate Governance", "issue": "Board", "name": "Female Directors", "unit": "count", "type": "performance", "source": "DEF14A", "component": True},
    "women_on_board_pct": {"pillar": "Governance", "theme": "Corporate Governance", "issue": "Board", "name": "Women on Board", "unit": "%", "type": "performance", "source": "DEF14A", "derived_from": ["board_female", "board_size"]},
    "board_meetings": {"pillar": "Governance", "theme": "Corporate Governance", "issue": "Board", "name": "Board Meeting Attendance", "unit": "%", "type": "performance", "source": "DEF14A"},

    # GOVERNANCE - Corporate Governance (Pay) - Source: DEF 14A
    "ceo_pay_ratio": {"pillar": "Governance", "theme": "Corporate Governance", "issue": "Pay", "name": "CEO Pay Ratio", "unit": "ratio", "type": "performance", "source": "DEF14A"},
    "say_on_pay": {"pillar": "Governance", "theme": "Corporate Governance", "issue": "Pay", "name": "Say-on-Pay Approval", "unit": "%", "type": "performance", "source": "DEF14A"},

    # GOVERNANCE - Corporate Governance (Ownership) - Source: DEF 14A
    "dual_class_shares": {"pillar": "Governance", "theme": "Corporate Governance", "issue": "Ownership", "name": "Dual-Class Shares", "unit": "Y/N", "type": "policy", "source": "DEF14A"},

    # GOVERNANCE - Corporate Behavior (Ethics) - Source: DEF 14A
    "ethics_policy": {"pillar": "Governance", "theme": "Corporate Behavior", "issue": "Ethics", "name": "Code of Ethics", "unit": "Y/N", "type": "policy", "source": "DEF14A"},
    "anti_corruption_policy": {"pillar": "Governance", "theme": "Corporate Behavior", "issue": "Ethics", "name": "Anti-Corruption Policy", "unit": "Y/N", "type": "policy", "source": "DEF14A"},
    "whistleblower_policy": {"pillar": "Governance", "theme": "Corporate Behavior", "issue": "Ethics", "name": "Whistleblower Policy", "unit": "Y/N", "type": "policy", "source": "DEF14A"},
    "ethics_violations": {"pillar": "Governance", "theme": "Corporate Behavior", "issue": "Ethics", "name": "Ethics Violations", "unit": "count", "type": "performance", "source": "DEF14A"},

    # GOVERNANCE - Corporate Behavior (Tax) - Source: 10-K
    "effective_tax_rate": {"pillar": "Governance", "theme": "Corporate Behavior", "issue": "Tax", "name": "Effective Tax Rate", "unit": "%", "type": "performance", "source": "10-K"},
    "country_by_country_reporting": {"pillar": "Governance", "theme": "Corporate Behavior", "issue": "Tax", "name": "Country-by-Country Reporting", "unit": "Y/N", "type": "policy", "source": "10-K"},

    # GOVERNANCE - Accounting - Source: 10-K
    "accounting_controversies": {"pillar": "Governance", "theme": "Corporate Governance", "issue": "Accounting", "name": "Accounting Controversies", "unit": "count", "type": "performance", "source": "10-K"},

    # ═══════════════════════════════════════════════════════════════════════
    # COMPONENT - Financial
    # ═══════════════════════════════════════════════════════════════════════
    "revenue": {"pillar": "Component", "theme": "Financial", "issue": "Financial Data", "name": "Total Revenue", "unit": "USD", "type": "performance", "component": True},
}

# ═══════════════════════════════════════════════════════════════════════════
# HELPER LISTS
# ═══════════════════════════════════════════════════════════════════════════

SUSTAINABILITY_METRICS = [k for k, v in ESG_METRICS.items() if v.get('source') not in ['DEF14A', '10-K']]
SEC_DEF14A_METRICS = [k for k, v in ESG_METRICS.items() if v.get('source') == 'DEF14A']
SEC_10K_METRICS = [k for k, v in ESG_METRICS.items() if v.get('source') == '10-K']

PERFORMANCE_METRICS = [k for k, v in ESG_METRICS.items() if v.get('type') == 'performance']
POLICY_METRICS = [k for k, v in ESG_METRICS.items() if v.get('type') == 'policy']

COMPONENT_METRICS = [k for k, v in ESG_METRICS.items() if v.get('component') == True]
DERIVED_METRICS = {k: v['derived_from'] for k, v in ESG_METRICS.items() if 'derived_from' in v}

print(f"✅ ESG_METRICS defined: {len(ESG_METRICS)} metrics")
print(f"\n   By Source:")
print(f"      Sustainability Reports: {len(SUSTAINABILITY_METRICS)}")
print(f"      DEF 14A: {len(SEC_DEF14A_METRICS)}")
print(f"      10-K: {len(SEC_10K_METRICS)}")
print(f"\n   By Type:")
print(f"      Performance: {len(PERFORMANCE_METRICS)}")
print(f"      Policy: {len(POLICY_METRICS)}")
print(f"\n   Special:")
print(f"      Component (for calculations): {len(COMPONENT_METRICS)}")
print(f"      Derived (calculable): {len(DERIVED_METRICS)}")

✅ ESG_METRICS defined: 51 metrics

   By Source:
      Sustainability Reports: 35
      DEF 14A: 13
      10-K: 3

   By Type:
      Performance: 44
      Policy: 7

   Special:
      Component (for calculations): 13
      Derived (calculable): 7


In [ ]:
l = []
for i in ESG_METRICS.keys():
  l.append(i)

l

['scope_1_emissions',
 'scope_2_emissions_market',
 'scope_2_emissions_location',
 'scope_3_emissions',
 'carbon_intensity',
 'climate_targets',
 'energy_total',
 'energy_renewable',
 'renewable_energy_pct',
 'water_withdrawal',
 'water_recycled',
 'waste_total',
 'waste_recycled',
 'hazardous_waste',
 'ewaste_generated',
 'ewaste_recycled',
 'green_building_certifications',
 'employees_total',
 'employee_turnover',
 'safety_hours_worked',
 'lost_time_injuries',
 'recordable_injuries',
 'ltifr',
 'trir',
 'fatalities',
 'training_hours_total',
 'training_hours_per_emp',
 'women_workforce',
 'women_management',
 'women_senior_mgmt',
 'supplier_audits',
 'supplier_code_of_conduct',
 'data_breaches',
 'community_investment',
 'board_size',
 'board_independent',
 'board_independence_pct',
 'board_female',
 'women_on_board_pct',
 'board_meetings',
 'ceo_pay_ratio',
 'say_on_pay',
 'dual_class_shares',
 'ethics_policy',
 'anti_corruption_policy',
 'whistleblower_policy',
 'ethics_violations'

In [ ]:
counts = {'Environment': 0,
          'Social': 0,
          'Governance': 0,
          'Component': 0}
for metric in ESG_METRICS:
  counts[ESG_METRICS[metric]['pillar']] += 1
print(counts)

{'Environment': 17, 'Social': 17, 'Governance': 16, 'Component': 1}


### Show Pillar/Theme/Issue

In [ ]:
pillars = set()
themes = set()
issues = set()
metrics = set()


for metric_id, cfg in ESG_METRICS.items():
    # print(f"Metric ID: {metric_id}")
    # print(f"cfg: {cfg}")
    pillar = cfg['pillar']
    theme = cfg['theme']
    issue = cfg['issue']
    pillars.add(pillar)
    themes.add(theme)
    issues.add(issue)
    metrics.add(metric_id)


print(f"Pillars: {len(pillars)}")
print(f"Themes: {len(themes)}")
print(f"Issues: {len(issues)}")
print(f"Metrics: {len(metrics)}")

Pillars: 4
Themes: 10
Issues: 21
Metrics: 51


## Utility Functions

In [ ]:
# =============================================================================
# PROMPTS - TUNE THESE FOR BETTER EXTRACTION
# =============================================================================

# Prompts moved to "Extraction Parameters"

def format_metrics_for_prompt(metrics: Dict) -> str:
    """Format metrics dictionary for prompt."""
    lines = []
    by_pillar = {}
    for metric_id, cfg in metrics.items():
        pillar = cfg['pillar']
        if pillar not in by_pillar:
            by_pillar[pillar] = []
        by_pillar[pillar].append((metric_id, cfg))

    for pillar in ['Component', 'Environment', 'Social', 'Governance']:
        if pillar in by_pillar:
            lines.append(f"\n{pillar.upper()}:")
            for metric_id, cfg in by_pillar[pillar]:
                lines.append(f"- {cfg['name']} (`{metric_id}`): {cfg['unit']}")

    return '\n'.join(lines)

print("✅ Prompts defined")

✅ Prompts defined


In [ ]:
# =============================================================================
# UTILITY FUNCTIONS
# =============================================================================

def get_year(filename: str) -> Optional[int]:
    """Extract year from filename."""
    for year in YEARS:
        if str(year) in filename:
            return year
    return None


def get_fiscal_year_def14a(filename: str) -> Optional[int]:
    """DEF 14A: filename year = fiscal year (manually downloaded)."""
    for year in range(2025, 2019, -1):
        if str(year) in filename:
            return year
    return None

# Updated: Manually changed all 10K documents so that filename year = fiscal year. No more -1 necessary
def get_fiscal_year_10k(filename: str, company: str) -> Optional[int]:
    """10-K: filename year = fiscal year (after manual clean up)."""
    for year in range(2025, 2019, -1):
        if str(year) in filename:
            return year
    return None


def get_applicable_metrics(sector: str) -> Dict:
    """Return all metrics (no sector filtering in V9)."""
    return ESG_METRICS


print("✅ Utility functions defined")

✅ Utility functions defined


In [ ]:
# =============================================================================
# JSON PARSING
# =============================================================================

def parse_json_response(text: str) -> List[Dict]:
    """
    Extract JSON array from LLM response.
    Handles markdown code blocks, truncation, and malformed responses.
    """
    if not text:
        return []

    text = text.strip()

    # Remove markdown code block wrapper
    if text.startswith('```'):
        text = re.sub(r'^```(?:json)?\s*\n?', '', text)
        text = re.sub(r'\n?```\s*$', '', text)
        text = text.strip()

    # Find JSON array start
    start_idx = text.find('[')
    if start_idx == -1:
        return []

    text = text[start_idx:]

    # Method 1: Direct parse
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Method 2: Find matching bracket
    bracket_count = 0
    end_idx = -1
    for i, char in enumerate(text):
        if char == '[':
            bracket_count += 1
        elif char == ']':
            bracket_count -= 1
            if bracket_count == 0:
                end_idx = i
                break

    if end_idx != -1:
        try:
            return json.loads(text[:end_idx + 1])
        except json.JSONDecodeError:
            pass

    # Method 3: Handle truncated JSON - find last complete object
    last_complete = text.rfind('},')
    if last_complete == -1:
        last_complete = text.rfind('}\n')
    if last_complete == -1:
        last_complete = text.rfind('}')

    if last_complete != -1:
        truncated = text[:last_complete + 1] + ']'
        try:
            results = json.loads(truncated)
            print(f"      ⚠️ Recovered {len(results)} values from truncated JSON")
            return results
        except json.JSONDecodeError:
            pass

    # Method 4: Extract individual objects via regex
    objects = []
    pattern = r'\{\s*"metric_id"\s*:\s*"[^"]+"\s*,\s*"value"\s*:\s*[^,}]+[^}]*\}'
    for match in re.finditer(pattern, text, re.DOTALL):
        try:
            obj_str = match.group()
            if not obj_str.rstrip().endswith('}'):
                continue
            obj = json.loads(obj_str)
            if 'metric_id' in obj and obj.get('value') is not None:
                objects.append(obj)
        except:
            pass

    if objects:
        print(f"      ⚠️ Extracted {len(objects)} values via regex")

    return objects


# =============================================================================
# DEDUPLICATION
# =============================================================================

def deduplicate_results(results: List[Dict]) -> List[Dict]:
    """
    Remove duplicates within same source file only.
    Cross-document deduplication happens in create_comparison.
    """
    unique = {}
    confidence_rank = {'high': 3, 'medium': 2, 'low': 1, 'calculated': 2, 'corrected': 2}

    for r in results:
        key = (r.get('metric_id'), r.get('year'), r.get('source_file'))
        if key not in unique:
            unique[key] = r
        else:
            existing_conf = confidence_rank.get(unique[key].get('confidence', 'low'), 0)
            new_conf = confidence_rank.get(r.get('confidence', 'low'), 0)
            if new_conf > existing_conf:
                unique[key] = r

    return list(unique.values())


print("✅ Deduplication updated (per-file only, preserves cross-document values)")

def validate_and_fix_results(results: List[Dict]) -> List[Dict]:
    """Fix common extraction errors."""
    fixed = []

    for r in results:
        r = r.copy()
        metric_id = r.get('metric_id')
        value = r.get('value')
        context = str(r.get('context', '')).lower()

        # Fix board_meetings: "all attended at least X%" should be X, not 100
        if metric_id == 'board_meetings' and value == 100:
            pct_match = re.search(r'(\d{2,3})\s*(?:percent|%)', context)
            if pct_match:
                actual = int(pct_match.group(1))
                if actual != 100:
                    r['value'] = actual
                    r['confidence'] = 'corrected'

        # Fix percentages > 100 (likely decimal errors)
        if r.get('unit') == '%' and value is not None:
            if value > 100 and metric_id not in ['ceo_pay_ratio']:
                if value > 1000:
                    r['value'] = value / 100

        fixed.append(r)

    return fixed

print("✅ JSON parsing functions defined")

✅ Deduplication updated (per-file only, preserves cross-document values)
✅ JSON parsing functions defined


In [ ]:
# =============================================================================
# EXTRACTION ANALYSIS & FILTERING
# =============================================================================

def analyze_extraction_loss(results: List[Dict], filename: str) -> Dict:
    """Analyze why extracted values might not make it to final output."""

    valid_years = set(YEARS)
    valid_metrics = set(ESG_METRICS.keys())

    issues = {
        'invalid_year': [],
        'invalid_metric': [],
        'missing_value': [],
        'valid': []
    }

    for r in results:
        metric_id = r.get('metric_id')
        year = r.get('year')
        value = r.get('value')

        if year not in valid_years:
            issues['invalid_year'].append(f"{metric_id} ({year})")
        elif metric_id not in valid_metrics:
            issues['invalid_metric'].append(f"{metric_id} ({year})")
        elif value is None:
            issues['missing_value'].append(f"{metric_id} ({year})")
        else:
            issues['valid'].append(f"{metric_id} ({year})")

    print(f"\n      📊 Extraction Analysis for {filename}:")
    print(f"         Valid:          {len(issues['valid'])}")
    print(f"         Invalid year:   {len(issues['invalid_year'])}")
    print(f"         Invalid metric: {len(issues['invalid_metric'])}")
    print(f"         Missing value:  {len(issues['missing_value'])}")

    if issues['invalid_year']:
        years = [r.get('year') for r in results if r.get('year') not in valid_years]
        year_counts = Counter(years)
        print(f"         Invalid years: {dict(year_counts)}")

    if issues['invalid_metric'][:5]:
        print(f"         Invalid metrics (first 5): {issues['invalid_metric'][:5]}")

    return issues


def filter_valid_results(results: List[Dict]) -> List[Dict]:
    """Filter out invalid extractions before saving."""

    valid_years = set(YEARS)
    valid_metrics = set(ESG_METRICS.keys())

    filtered = []
    removed = {'year': 0, 'metric': 0, 'value': 0, 'duplicate': 0}
    seen_keys = set()

    for r in results:
        metric_id = r.get('metric_id')
        year = r.get('year')
        value = r.get('value')

        # Skip invalid years
        if year not in valid_years:
            removed['year'] += 1
            continue

        # Skip invalid metric_ids
        if metric_id not in valid_metrics:
            removed['metric'] += 1
            continue

        # Skip null values
        if value is None:
            removed['value'] += 1
            continue

        # Skip duplicates (keep first)
        key = (metric_id, year)
        if key in seen_keys:
            removed['duplicate'] += 1
            continue

        seen_keys.add(key)
        filtered.append(r)

    total_removed = sum(removed.values())
    if total_removed > 0:
        print(f"      🗑️ Filtered out {total_removed}: year={removed['year']}, metric={removed['metric']}, value={removed['value']}, dup={removed['duplicate']}")

    return filtered

print("✅ Analysis and filtering functions defined")

✅ Analysis and filtering functions defined


In [ ]:
def compress_pdf_for_api(pdf_path: str, max_mb: float = 9.0) -> bytes:
    """Compress a PDF by rendering pages as images if it exceeds size limit."""
    with open(pdf_path, 'rb') as f:
        pdf_bytes = f.read()

    if len(pdf_bytes) / (1024 * 1024) <= max_mb:
        return pdf_bytes

    print(f"         📸 Compressing large PDF...")
    doc = fitz.open(pdf_path)
    new_doc = fitz.open()

    for page in doc:
        scale = 1200 / page.rect.width
        pix = page.get_pixmap(matrix=fitz.Matrix(scale, scale))
        img_bytes = pix.tobytes("jpeg", jpg_quality=75)
        new_page = new_doc.new_page(width=pix.width, height=pix.height)
        new_page.insert_image(new_page.rect, stream=img_bytes)

    compressed = new_doc.tobytes(garbage=4, deflate=True)
    new_doc.close()
    doc.close()

    print(f"         📸 Compressed: {len(pdf_bytes)/(1024*1024):.1f} MB → {len(compressed)/(1024*1024):.1f} MB")
    return compressed

## Extraction Functions

In [ ]:
# =============================================================================
# SUSTAINABILITY PDF EXTRACTION (WITH STREAMING)
# =============================================================================

def extract_from_pdf_single(pdf_bytes: bytes, filename: str, prompt: str,
                             doc_type: str) -> Tuple[List[Dict], int, int]:
    """Send a single PDF (as bytes) to the API. Returns (results, input_tokens, output_tokens)."""
    pdf_base64 = base64.standard_b64encode(pdf_bytes).decode('utf-8')

    full_response = ""
    with client.messages.stream(
        model=API_MODEL,
        max_tokens=EXTRACTION_CONFIG['max_tokens_output'],
        messages=[{
            "role": "user",
            "content": [
                {"type": "document", "source": {"type": "base64", "media_type": "application/pdf", "data": pdf_base64}},
                {"type": "text", "text": prompt}
            ]
        }]
    ) as stream:
        for text in stream.text_stream:
            full_response += text
        final_message = stream.get_final_message()
        input_tokens = final_message.usage.input_tokens
        output_tokens = final_message.usage.output_tokens

    results = parse_json_response(full_response)
    results = validate_and_fix_results(results)

    if output_tokens >= EXTRACTION_CONFIG['max_tokens_output'] - 1000:
        print(f"         ⚠️ High output ({output_tokens} tokens) — may be truncated")

    return results, input_tokens, output_tokens


def extract_from_pdf(pdf_path: str, company: str, prompt: str, doc_type: str,
                     max_pages_per_chunk: int = 50, max_mb_per_chunk: float = 9.0) -> Tuple[List[Dict], Dict]:
    """
    Extract metrics from a PDF using Claude API.
    Automatically chunks if PDF exceeds 100-page or size limits.
    """
    filename = Path(pdf_path).name
    print(f"      📄 {filename}")

    # Get page count and file size
    doc = fitz.open(pdf_path)
    total_pages = len(doc)
    doc.close()

    with open(pdf_path, 'rb') as f:
        file_size_mb = len(f.read()) / (1024 * 1024)
    print(f"         Size: {file_size_mb:.2f} MB, {total_pages} pages")

    # Decide: single call or chunked
    needs_chunking = total_pages > max_pages_per_chunk or file_size_mb > max_mb_per_chunk

    if not needs_chunking:
        # Single call
        pdf_bytes = compress_pdf_for_api(pdf_path, max_mb_per_chunk)
        estimated_tokens = int(len(pdf_bytes) / (1024 * 1024) * 15000)
        rate_limiter.wait_if_needed(estimated_tokens)

        try:
            results, input_tokens, output_tokens = extract_from_pdf_single(
                pdf_bytes, filename, prompt, doc_type
            )
            rate_limiter.add_tokens(input_tokens)

            analyze_extraction_loss(results, filename)
            results = filter_valid_results(results)

            for r in results:
                r['source_file'] = filename
                r['source_type'] = doc_type

            results = deduplicate_results(results)
            print(f"         ✅ {len(results)} values ({input_tokens:,} in / {output_tokens:,} out)")

            return results, {'filename': filename, 'input_tokens': input_tokens, 'output_tokens': output_tokens}

        except Exception as e:
            error_msg = str(e)
            print(f"         ❌ Error: {error_msg[:300]}")

            if "rate_limit" in error_msg.lower():
                print(f"         ⏳ Rate limited, waiting {EXTRACTION_CONFIG['retry_wait_seconds']}s...")
                time.sleep(EXTRACTION_CONFIG['retry_wait_seconds'])
                return extract_from_pdf(pdf_path, company, prompt, doc_type)

            return [], {'filename': filename, 'error': error_msg[:300], 'input_tokens': 0, 'output_tokens': 0}

    else:
        # Chunked extraction
        doc = fitz.open(pdf_path)
        total_pages = len(doc)

        # Build chunks
        chunks = []
        for start in range(0, total_pages, max_pages_per_chunk):
            end = min(start + max_pages_per_chunk, total_pages)
            chunks.append((start, end))

        print(f"         📦 Chunking: {total_pages} pages → {len(chunks)} chunks of ≤{max_pages_per_chunk} pages")

        all_results = []
        total_input_tokens = 0
        total_output_tokens = 0

        for chunk_idx, (start, end) in enumerate(chunks):
            # Extract chunk as separate PDF
            chunk_doc = fitz.open()
            chunk_doc.insert_pdf(doc, from_page=start, to_page=end - 1)
            chunk_bytes = chunk_doc.tobytes(garbage=4, deflate=True)
            chunk_doc.close()

            chunk_mb = len(chunk_bytes) / (1024 * 1024)

            # Compress if needed
            if chunk_mb > max_mb_per_chunk:
                temp_path = f"/tmp/chunk_{chunk_idx}.pdf"
                with open(temp_path, 'wb') as f:
                    f.write(chunk_bytes)
                chunk_bytes = compress_pdf_for_api(temp_path, max_mb_per_chunk)
                chunk_mb = len(chunk_bytes) / (1024 * 1024)

            print(f"         📦 Chunk {chunk_idx + 1}/{len(chunks)}: pages {start + 1}-{end} ({chunk_mb:.1f} MB)")

            estimated_tokens = int(chunk_mb * 15000)
            rate_limiter.wait_if_needed(estimated_tokens)

            try:
                results, input_tokens, output_tokens = extract_from_pdf_single(
                    chunk_bytes, filename, prompt, doc_type
                )
                rate_limiter.add_tokens(input_tokens)
                total_input_tokens += input_tokens
                total_output_tokens += output_tokens

                results = filter_valid_results(results)

                # Adjust source_page for chunk offset
                for r in results:
                    r['source_file'] = filename
                    r['source_type'] = doc_type
                    if r.get('source_page'):
                        r['source_page'] = r['source_page'] + start

                all_results.extend(results)
                print(f"            ✅ {len(results)} values ({input_tokens:,} in / {output_tokens:,} out)")

            except Exception as e:
                error_msg = str(e)
                print(f"            ❌ Error: {error_msg[:300]}")

                if "rate_limit" in error_msg.lower():
                    print(f"            ⏳ Rate limited, waiting {EXTRACTION_CONFIG['retry_wait_seconds']}s...")
                    time.sleep(EXTRACTION_CONFIG['retry_wait_seconds'])

            time.sleep(2)

        doc.close()

        all_results = deduplicate_results(all_results)
        print(f"         ✅ Total: {len(all_results)} values after dedup ({total_input_tokens:,} in / {total_output_tokens:,} out)")

        return all_results, {
            'filename': filename,
            'chunks': len(chunks),
            'input_tokens': total_input_tokens,
            'output_tokens': total_output_tokens
        }



def extract_from_pdf_chunked(pdf_path: str, company: str, prompt: str, doc_type: str,
                              max_chunk_mb: float = 9.0) -> Tuple[List[Dict], Dict]:
    """
    Split a large PDF into chunks and extract from each chunk separately.
    Results are merged and deduplicated across chunks.
    """
    filename = Path(pdf_path).name

    doc = fitz.open(pdf_path)
    total_pages = len(doc)

    # Estimate pages per chunk based on average page size
    with open(pdf_path, 'rb') as f:
        total_bytes = len(f.read())
    avg_page_mb = (total_bytes / (1024 * 1024)) / total_pages
    pages_per_chunk = max(10, int(max_chunk_mb / avg_page_mb * 0.7))  # 70% safety margin

    # Build chunks
    chunks = []
    for start in range(0, total_pages, pages_per_chunk):
        end = min(start + pages_per_chunk, total_pages)
        chunks.append((start, end))

    print(f"         📦 Chunking: {total_pages} pages → {len(chunks)} chunks of ~{pages_per_chunk} pages")

    all_results = []
    total_input_tokens = 0
    total_output_tokens = 0

    for chunk_idx, (start, end) in enumerate(chunks):
        # Extract chunk as separate PDF
        chunk_doc = fitz.open()
        chunk_doc.insert_pdf(doc, from_page=start, to_page=end - 1)
        chunk_bytes = chunk_doc.tobytes(garbage=4, deflate=True)
        chunk_doc.close()

        chunk_mb = len(chunk_bytes) / (1024 * 1024)

        # Compress chunk if still too large
        if chunk_mb > max_chunk_mb:
            temp_path = f"/tmp/chunk_{chunk_idx}.pdf"
            with open(temp_path, 'wb') as f:
                f.write(chunk_bytes)
            chunk_bytes = compress_pdf_for_api(temp_path, max_chunk_mb)
            chunk_mb = len(chunk_bytes) / (1024 * 1024)

        print(f"         📦 Chunk {chunk_idx + 1}/{len(chunks)}: pages {start + 1}-{end} ({chunk_mb:.1f} MB)")

        estimated_tokens = int(chunk_mb * 15000)
        rate_limiter.wait_if_needed(estimated_tokens)

        pdf_base64 = base64.standard_b64encode(chunk_bytes).decode('utf-8')

        try:
            full_response = ""
            with client.messages.stream(
                model=API_MODEL,
                max_tokens=EXTRACTION_CONFIG['max_tokens_output'],
                messages=[{
                    "role": "user",
                    "content": [
                        {"type": "document", "source": {"type": "base64", "media_type": "application/pdf", "data": pdf_base64}},
                        {"type": "text", "text": prompt}
                    ]
                }]
            ) as stream:
                for text in stream.text_stream:
                    full_response += text
                final_message = stream.get_final_message()
                input_tokens = final_message.usage.input_tokens
                output_tokens = final_message.usage.output_tokens

            rate_limiter.add_tokens(input_tokens)
            total_input_tokens += input_tokens
            total_output_tokens += output_tokens

            results = parse_json_response(full_response)
            results = validate_and_fix_results(results)
            results = filter_valid_results(results)

            # Adjust source_page to account for chunk offset
            for r in results:
                r['source_file'] = filename
                r['source_type'] = doc_type
                if r.get('source_page'):
                    try:
                        r['source_page'] = int(r['source_page']) + start
                    except (ValueError, TypeError):
                        pass

            all_results.extend(results)
            print(f"            ✅ {len(results)} values ({input_tokens:,} in / {output_tokens:,} out)")

        except Exception as e:
            error_msg = str(e)
            print(f"            ❌ Error: {error_msg[:100]}")

            if "rate_limit" in error_msg.lower():
                print(f"            ⏳ Rate limited, waiting {EXTRACTION_CONFIG['retry_wait_seconds']}s...")
                time.sleep(EXTRACTION_CONFIG['retry_wait_seconds'])
                # Retry this chunk
                chunk_idx -= 1
                continue

        time.sleep(2)

    doc.close()

    # Deduplicate across chunks
    all_results = deduplicate_results(all_results)
    print(f"         ✅ Total: {len(all_results)} values after dedup ({total_input_tokens:,} in / {total_output_tokens:,} out)")

    return all_results, {
        'filename': filename,
        'chunks': len(chunks),
        'input_tokens': total_input_tokens,
        'output_tokens': total_output_tokens
    }

print("✅ PDF extraction with chunking defined")

✅ PDF extraction with chunking defined


In [ ]:
# =============================================================================
# SUSTAINABILITY WRAPPER — CDP DETECTION + SEPARATE PROMPT
# =============================================================================

PROMPT_CDP = """Extract environmental metrics from this CDP climate questionnaire for {company}.

METRICS TO EXTRACT ({metric_count} total):
{metrics}

═══════════════════════════════════════════════════════════════════════════════
STRICT RULES:
═══════════════════════════════════════════════════════════════════════════════
- ONLY extract for years 2020, 2021, 2022, 2023, 2024
- ONLY use metric_id values from the list above
- Extract the EXACT unit as shown in the document (MWh, GWh, GJ, etc.)
- Extract values ONLY from explicit answers and data tables, NOT from charts
- IGNORE goals, targets, projections or %-reductions comparing to previous years
- IGNORE single country spotlights and country-specific data for total/absolute values
- CDP section references (C6.1, C8.2a, etc.) indicate where data is located
- EVERY entry MUST include "context" with the CDP section ID or table name

OUTPUT — Return ONLY a JSON array:
[
  {{"metric_id": "scope_1_emissions", "value": 6568, "unit": "tCO2e", "year": 2023, "confidence": "high", "context": "C6.1 Scope 1 emissions"}}
]

If no relevant data found, return: []
"""

# Environmental-only metrics for CDP
CDP_METRIC_IDS = [
    'scope_1_emissions', 'scope_2_emissions_market', 'scope_2_emissions_location',
    'scope_3_emissions', 'carbon_intensity', 'climate_targets',
    'energy_total', 'energy_renewable', 'renewable_energy_pct',
    'water_withdrawal', 'water_recycled',
    'waste_total', 'waste_recycled', 'hazardous_waste',
]

def extract_sustainability_pdfs(company: str, company_dir: Path) -> Tuple[List[Dict], Dict]:
    """Extract from sustainability PDFs with CDP detection."""
    print(f"\n   📄 Sustainability PDFs")

    all_results = []
    total_input_tokens = 0
    total_output_tokens = 0

    if USE_FILTERED:
        pdf_files = [f for f in company_dir.glob("FILTERED_Report_*.pdf")
                     if get_year(f.name) in YEARS]
    else:
        pdf_files = [f for f in company_dir.glob("*.pdf")
                     if get_year(f.name) in YEARS
                     and not any(x in f.name for x in ['10-K', '10K', 'DEF_14A', 'DEF14A'])]

    pdf_files = sorted(pdf_files, key=lambda f: get_year(f.name) or 0, reverse=True)
    print(f"      Found: {len(pdf_files)} files")

    if not pdf_files:
        return [], {'input_tokens': 0, 'output_tokens': 0}

    # Full metrics for standard reports
    all_metrics = get_applicable_metrics(SECTOR)
    all_metrics_text = format_metrics_for_prompt(all_metrics)

    # Reduced metrics for CDP
    cdp_metrics = {k: v for k, v in all_metrics.items() if k in CDP_METRIC_IDS}
    cdp_metrics_text = format_metrics_for_prompt(cdp_metrics)

    for pdf_path in pdf_files:
        # Detect if CDP
        is_cdp = 'cdp' in pdf_path.name.lower()
        if not is_cdp:
            # Quick content check on first pages
            try:
                doc = fitz.open(str(pdf_path))
                first_text = ""
                for i in range(min(5, len(doc))):
                    first_text += doc[i].get_text().lower()
                doc.close()
                cdp_signals = ['cdp climate change', 'cdp questionnaire', 'cdp response', 'c0.1', 'c1.1']
                if sum(1 for s in cdp_signals if s in first_text) >= 2:
                    is_cdp = True
            except:
                pass

        if is_cdp:
            print(f"      📋 CDP detected: {pdf_path.name}")
            prompt = PROMPT_CDP.format(
                company=company,
                metric_count=len(cdp_metrics),
                metrics=cdp_metrics_text
            )
        else:
            prompt = PROMPT_SUSTAINABILITY.format(
                company=company,
                metric_count=len(all_metrics),
                metrics=all_metrics_text
            )

        results, meta = extract_from_pdf(str(pdf_path), company, prompt, 'sustainability_pdf')
        all_results.extend(results)
        total_input_tokens += meta.get('input_tokens', 0)
        total_output_tokens += meta.get('output_tokens', 0)
        time.sleep(2)

    print(f"      Total: {len(all_results)} values")
    print(f"      Tokens: {total_input_tokens:,} in / {total_output_tokens:,} out")

    return all_results, {
        'input_tokens': total_input_tokens,
        'output_tokens': total_output_tokens
    }

print("✅ Extraction wrappers updated (no year override, CDP detection)")

✅ Extraction wrappers updated (no year override, CDP detection)


In [ ]:
# =============================================================================
# EXTRACTION WRAPPER FUNCTIONS — UPDATED FOR DUAL MODE
# =============================================================================

def extract_sustainability_pdfs(company: str, company_dir: Path) -> Tuple[List[Dict], Dict]:
    """Extract from sustainability PDFs (filtered or unfiltered)."""
    print(f"\n   📄 Sustainability PDFs")

    all_results = []
    total_input_tokens = 0
    total_output_tokens = 0

    if USE_FILTERED:
        # Filtered mode: look for FILTERED_ prefix
        pdf_files = [f for f in company_dir.glob("FILTERED_Report_*.pdf")
                     if get_year(f.name) in YEARS]
    else:
        # Unfiltered mode: all report PDFs (exclude SEC filings)
        pdf_files = [f for f in company_dir.glob("*.pdf")
                     if get_year(f.name) in YEARS
                     and not any(x in f.name for x in ['10-K', '10K', 'DEF_14A', 'DEF14A'])]

    pdf_files = sorted(pdf_files, key=lambda f: get_year(f.name) or 0, reverse=True)
    print(f"      Found: {len(pdf_files)} files")

    if not pdf_files:
        return [], {'input_tokens': 0, 'output_tokens': 0}

    metrics = get_applicable_metrics(SECTOR)
    metrics_text = format_metrics_for_prompt(metrics)
    prompt = PROMPT_SUSTAINABILITY.format(
        company=company,
        metric_count=len(metrics),
        metrics=metrics_text
    )

    for pdf_path in pdf_files:
        results, meta = extract_from_pdf(str(pdf_path), company, prompt, 'sustainability_pdf')
        all_results.extend(results)
        total_input_tokens += meta.get('input_tokens', 0)
        total_output_tokens += meta.get('output_tokens', 0)
        time.sleep(2)

    print(f"      Total: {len(all_results)} values")
    print(f"      Tokens: {total_input_tokens:,} in / {total_output_tokens:,} out")

    return all_results, {
        'input_tokens': total_input_tokens,
        'output_tokens': total_output_tokens
    }


def extract_def14a_pdfs(company: str, company_dir: Path) -> Tuple[List[Dict], Dict]:
    """Extract from DEF 14A PDFs."""
    print(f"\n   📋 DEF 14A PDFs")

    all_results = []
    total_input_tokens = 0
    total_output_tokens = 0

    if USE_FILTERED:
        pdf_files = (list(company_dir.glob("FILTERED_*DEF_14A*.pdf")) +
                     list(company_dir.glob("FILTERED_*DEF14A*.pdf")))
    else:
        pdf_files = (list(company_dir.glob("*DEF_14A*.pdf")) +
                     list(company_dir.glob("*DEF14A*.pdf")))

    print(f"      Found: {len(pdf_files)} files")

    for pdf_path in sorted(pdf_files, key=lambda f: get_year(f.name) or 0, reverse=True):
        fiscal_year = get_fiscal_year_def14a(pdf_path.name)
        if not fiscal_year or fiscal_year not in YEARS:
            continue

        prompt = PROMPT_DEF14A.format(company=company, fiscal_year=fiscal_year)
        results, meta = extract_from_pdf(str(pdf_path), company, prompt, 'sec_def14a')

        # Only set year as fallback if LLM didn't provide one
        for r in results:
            if not r.get('year') or r['year'] not in YEARS:
                r['year'] = fiscal_year

        all_results.extend(results)
        total_input_tokens += meta.get('input_tokens', 0)
        total_output_tokens += meta.get('output_tokens', 0)
        time.sleep(2)

    print(f"      Total: {len(all_results)} values")
    print(f"      Tokens: {total_input_tokens:,} in / {total_output_tokens:,} out")

    return all_results, {
        'input_tokens': total_input_tokens,
        'output_tokens': total_output_tokens
    }


def extract_10k_pdfs(company: str, company_dir: Path) -> Tuple[List[Dict], Dict]:
    """Extract from 10-K PDFs."""
    print(f"\n   📋 10-K PDFs")

    all_results = []
    total_input_tokens = 0
    total_output_tokens = 0

    if USE_FILTERED:
        pdf_files = (list(company_dir.glob("FILTERED_*10-K*.pdf")) +
                     list(company_dir.glob("FILTERED_*10K*.pdf")))
    else:
        pdf_files = (list(company_dir.glob("*10-K*.pdf")) +
                     list(company_dir.glob("*10K*.pdf")))

    print(f"      Found: {len(pdf_files)} files")

    for pdf_path in sorted(pdf_files, key=lambda f: get_year(f.name) or 0, reverse=True):
        fiscal_year = get_fiscal_year_10k(pdf_path.name, company)
        if not fiscal_year or fiscal_year not in YEARS:
            continue

        prompt = PROMPT_10K.format(company=company, fiscal_year=fiscal_year)
        results, meta = extract_from_pdf(str(pdf_path), company, prompt, 'sec_10k')

        # Only set year as fallback if LLM didn't provide one
        for r in results:
            if not r.get('year') or r['year'] not in YEARS:
                r['year'] = fiscal_year

        all_results.extend(results)
        total_input_tokens += meta.get('input_tokens', 0)
        total_output_tokens += meta.get('output_tokens', 0)
        time.sleep(2)

    print(f"      Total: {len(all_results)} values")
    print(f"      Tokens: {total_input_tokens:,} in / {total_output_tokens:,} out")

    return all_results, {
        'input_tokens': total_input_tokens,
        'output_tokens': total_output_tokens
    }


print("✅ Extraction wrapper functions defined")
print(f"   Mode: {'FILTERED' if USE_FILTERED else 'UNFILTERED'}")

✅ Extraction wrapper functions defined
   Mode: FILTERED


## Merge and Export Functions

In [ ]:
# =============================================================================
# COMPARISON — SOURCE PRIORITY + RECENCY
# =============================================================================

def _extract_file_year(source_file: str) -> int:
    """Extract year from source filename for recency sorting."""
    if not source_file:
        return 0
    for year in range(2025, 2019, -1):
        if str(year) in source_file:
            return year
    return 0


def create_comparison(company: str, pdf_results: List[Dict], sec_results: List[Dict]) -> pd.DataFrame:
    """
    Create comparison with source priority:
    1. Preferred document type (SEC for governance, PDF for environmental/social)
    2. Within same type: newer source file preferred
    3. Within same file: higher confidence preferred
    """
    by_key = {}
    for r in pdf_results:
        key = (r.get('metric_id'), r.get('year'))
        if key[0] and key[1]:
            if key not in by_key:
                by_key[key] = {'pdf': [], 'sec': []}
            by_key[key]['pdf'].append(r)

    for r in sec_results:
        key = (r.get('metric_id'), r.get('year'))
        if key[0] and key[1]:
            if key not in by_key:
                by_key[key] = {'pdf': [], 'sec': []}
            by_key[key]['sec'].append(r)

    confidence_rank = {'high': 3, 'medium': 2, 'low': 1, 'calculated': 2}

    def pick_best(values_list):
        """Pick the best value: newest source file first, then highest confidence."""
        if not values_list:
            return None
        return max(values_list, key=lambda x: (
            _extract_file_year(x.get('source_file', '')),
            confidence_rank.get(x.get('confidence', 'low'), 0)
        ))

    rows = []
    for metric_id, cfg in ESG_METRICS.items():
        for year in YEARS:
            key = (metric_id, year)
            ext = by_key.get(key, {'pdf': [], 'sec': []})
            pdf_v, sec_v = ext['pdf'], ext['sec']

            # Pick best within each source type (newest + highest confidence)
            best_pdf = pick_best(pdf_v)
            best_sec = pick_best(sec_v)

            # Source priority: SEC for governance/tax, PDF for everything else
            final_value, final_source = None, None
            if metric_id in SEC_DEF14A_METRICS + SEC_10K_METRICS:
                if best_sec:
                    final_value = best_sec['value']
                    final_source = 'sec'
                elif best_pdf:
                    final_value = best_pdf['value']
                    final_source = 'pdf'
            else:
                if best_pdf:
                    final_value = best_pdf['value']
                    final_source = 'pdf'
                elif best_sec:
                    final_value = best_sec['value']
                    final_source = 'sec'

            # Build all alternatives for review
            all_values = []
            for r in sorted(pdf_v, key=lambda x: -_extract_file_year(x.get('source_file', ''))):
                all_values.append(
                    f"PDF: {r['value']} {r.get('unit','')} [{r.get('confidence','')}] "
                    f"({r.get('source_file','')}) - {r.get('context','')}"
                )
            for r in sorted(sec_v, key=lambda x: -_extract_file_year(x.get('source_file', ''))):
                all_values.append(
                    f"SEC: {r['value']} {r.get('unit','')} [{r.get('confidence','')}] "
                    f"({r.get('source_file','')}) - {r.get('context','')}"
                )

            rows.append({
                'company': company, 'metric_id': metric_id, 'metric_name': cfg['name'],
                'year': year, 'pillar': cfg['pillar'], 'unit': cfg['unit'],
                'pdf_value': best_pdf['value'] if best_pdf else None,
                'pdf_unit': best_pdf.get('unit') if best_pdf else None,
                'pdf_conf': best_pdf.get('confidence') if best_pdf else None,
                'pdf_source': best_pdf.get('source_file') if best_pdf else None,
                'pdf_context': best_pdf.get('context') if best_pdf else None,
                'sec_value': best_sec['value'] if best_sec else None,
                'sec_unit': best_sec.get('unit') if best_sec else None,
                'sec_conf': best_sec.get('confidence') if best_sec else None,
                'sec_source': best_sec.get('source_file') if best_sec else None,
                'sec_context': best_sec.get('context') if best_sec else None,
                'final_value': final_value, 'final_source': final_source,
                'all_sources': ' | '.join(all_values) if all_values else None,
                'source_count': len(pdf_v) + len(sec_v),
            })

    return pd.DataFrame(rows)


print("✅ Comparison updated (source type priority + recency)")


# =============================================================================
# FIX 5: DERIVED METRICS — ONLY SIMPLE BOARD CALCULATIONS
# =============================================================================

def calculate_derived(results: List[Dict]) -> List[Dict]:
    """
    Calculate derived metrics from components.
    Only board metrics (simple count divisions) — skip energy/training
    due to unit mismatch risk.
    """
    by_year = {}
    for r in results:
        year = r.get('year')
        if year:
            if year not in by_year:
                by_year[year] = {}
            by_year[year][r.get('metric_id')] = r.get('value')

    derived = []
    for year, m in by_year.items():
        board_size = m.get('board_size')
        board_indep = m.get('board_independent')
        board_female = m.get('board_female')

        # Board independence %
        if board_size and board_indep and board_size > 0:
            derived.append({
                'metric_id': 'board_independence_pct',
                'value': round((board_indep / board_size) * 100, 1),
                'unit': '%', 'year': year, 'confidence': 'calculated',
                'context': f'{int(board_indep)}/{int(board_size)} independent',
                'source_type': 'calculated', 'source_file': 'derived'
            })

        # Women on board %
        if board_size and board_female and board_size > 0:
            derived.append({
                'metric_id': 'women_on_board_pct',
                'value': round((board_female / board_size) * 100, 1),
                'unit': '%', 'year': year, 'confidence': 'calculated',
                'context': f'{int(board_female)}/{int(board_size)} female',
                'source_type': 'calculated', 'source_file': 'derived'
            })

        # NOTE: Skipping these until unit normalization is implemented:
        # - renewable_energy_pct (energy_renewable / energy_total)
        # - training_hours_per_emp (training_hours_total / employees_total)
        # - carbon_intensity (scope_1 + scope_2 / revenue)
        # - ltifr, trir (injuries / hours worked)

    return derived


print("✅ Derived metrics updated (board only, energy/training deferred)")


def create_template(company: str) -> pd.DataFrame:
    """Create template with all metrics × years."""
    rows = []
    for metric_id, cfg in ESG_METRICS.items():
        for year in YEARS:
            rows.append({
                'company': company, 'sector': SECTOR, 'metric_id': metric_id,
                'year': year, 'metric_name': cfg['name'], 'pillar': cfg['pillar'],
                'theme': cfg['theme'], 'issue': cfg['issue'], 'expected_unit': cfg['unit'],
                'value': None, 'unit': None, 'source_file': None, 'source_type': None,
                'confidence': None, 'context': None, 'is_filled': False
            })
    return pd.DataFrame(rows)


def fill_template(template: pd.DataFrame, comparison: pd.DataFrame) -> pd.DataFrame:
    """Fill template from comparison final values."""
    df = template.copy()
    for _, row in comparison.iterrows():
        if pd.isna(row['final_value']):
            continue
        mask = (df['metric_id'] == row['metric_id']) & (df['year'] == row['year'])
        if mask.any():
            src = row['final_source']
            df.loc[mask, 'value'] = row['final_value']
            df.loc[mask, 'unit'] = row[f'{src}_unit'] if row.get(f'{src}_unit') else row['unit']
            df.loc[mask, 'source_type'] = src
            df.loc[mask, 'source_file'] = row[f'{src}_source']
            df.loc[mask, 'confidence'] = row[f'{src}_conf']
            df.loc[mask, 'context'] = row[f'{src}_context']
            df.loc[mask, 'is_filled'] = True
    return df


print("✅ Template functions defined")


✅ Comparison updated (source type priority + recency)
✅ Derived metrics updated (board only, energy/training deferred)
✅ Template functions defined


In [ ]:
# =============================================================================
# EXPORT — ADD "All Sources" SHEET FOR REVIEW
# =============================================================================

def export_results(df: pd.DataFrame, comparison: pd.DataFrame, company: str, output_dir: Path):
    """Export results with 7-sheet summary including all source values for review."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    filled = df['is_filled'].sum()
    total = len(df)
    pct = round(filled / total * 100, 1) if total > 0 else 0

    version_tag = f"v10_{RUN_LABEL}"

    # Main data files
    df.to_excel(output_dir / f"{company}_esg_{version_tag}.xlsx", index=False)
    df.to_csv(output_dir / f"{company}_esg_{version_tag}.csv", index=False)
    comparison.to_excel(output_dir / f"{company}_comparison_{version_tag}.xlsx", index=False)

    # ── Summary workbook ──
    # Sheet 1: Overview
    overview_data = pd.DataFrame({
        'Metric': ['Company', 'Total Metrics', 'Filled Metrics', 'Overall Coverage %',
                    'Model', 'Run Label', 'Input Mode'],
        'Value': [company, total, int(filled), f"{pct}%",
                  API_MODEL, RUN_LABEL, 'Filtered' if USE_FILTERED else 'Unfiltered']
    })

    # Sheet 2: Coverage by Year
    year_rows = []
    for year in sorted(df['year'].unique()):
        ydf = df[df['year'] == year]
        year_rows.append({
            'year': year,
            'total_metrics': len(ydf),
            'filled_metrics': int(ydf['is_filled'].sum()),
            'coverage_pct': round(ydf['is_filled'].mean() * 100, 1)
        })
    year_cov_df = pd.DataFrame(year_rows)

    # Sheet 3: Coverage by Pillar
    pillar_rows = []
    for pillar in ['Environment', 'Social', 'Governance', 'Component']:
        pdf = df[df['pillar'] == pillar]
        if len(pdf) > 0:
            pillar_rows.append({
                'pillar': pillar,
                'total_metrics': len(pdf),
                'filled_metrics': int(pdf['is_filled'].sum()),
                'coverage_pct': round(pdf['is_filled'].mean() * 100, 1)
            })
    pillar_cov_df = pd.DataFrame(pillar_rows)

    # Sheet 4: Filled Metrics
    filled_df = df[df['is_filled'] == True][
        ['metric_id', 'year', 'pillar', 'value', 'unit', 'source_type', 'confidence', 'context', 'source_file']
    ].sort_values(['pillar', 'metric_id', 'year'])

    # Sheet 5: Gaps
    gaps_df = df[df['is_filled'] == False][
        ['metric_id', 'year', 'pillar', 'theme', 'issue']
    ].sort_values(['pillar', 'metric_id', 'year'])

    # Sheet 6: Sources by Year
    filled_data = df[df['is_filled'] == True]
    if len(filled_data) > 0:
        sources_pivot = filled_data.groupby(['source_type', 'year']).size().unstack(fill_value=0)
        sources_df = sources_pivot.reset_index()
    else:
        sources_df = pd.DataFrame()

    # Sheet 7: All Sources (for review — shows every extracted value, not just the winner)
    multi_source = comparison[comparison['source_count'] > 0][[
        'metric_id', 'metric_name', 'year', 'pillar',
        'pdf_value', 'pdf_unit', 'pdf_conf', 'pdf_source', 'pdf_context',
        'sec_value', 'sec_unit', 'sec_conf', 'sec_source', 'sec_context',
        'final_value', 'final_source', 'source_count', 'all_sources'
    ]].sort_values(['pillar', 'metric_id', 'year'])

    # Write
    summary_path = output_dir / f"{company}_esg_{version_tag}_summary.xlsx"
    with pd.ExcelWriter(summary_path, engine='openpyxl') as w:
        overview_data.to_excel(w, sheet_name='Overview', index=False)
        year_cov_df.to_excel(w, sheet_name='Coverage by Year', index=False)
        pillar_cov_df.to_excel(w, sheet_name='Coverage by Pillar', index=False)
        filled_df.to_excel(w, sheet_name='Filled Metrics', index=False)
        gaps_df.to_excel(w, sheet_name='Gaps', index=False)
        if len(sources_df) > 0:
            sources_df.to_excel(w, sheet_name='Sources by Year', index=False)
        if len(multi_source) > 0:
            multi_source.to_excel(w, sheet_name='All Sources', index=False)

    # Print summary
    print(f"\n   ✅ Exported to: {output_dir}")
    print(f"      Coverage: {int(filled)}/{total} ({pct}%)")
    print(f"\n      By Year:")
    for _, r in year_cov_df.iterrows():
        print(f"         {int(r['year'])}: {r['coverage_pct']}% ({int(r['filled_metrics'])}/{int(r['total_metrics'])})")
    print(f"\n      By Pillar:")
    for _, r in pillar_cov_df.iterrows():
        print(f"         {r['pillar']}: {r['coverage_pct']}%")
    print(f"\n      Gaps: {len(gaps_df)} missing metric-year combinations")

    # Flag potential unit mismatches
    unit_issues = []
    for _, row in df[df['is_filled'] == True].iterrows():
        expected = ESG_METRICS.get(row['metric_id'], {}).get('unit', '')
        actual = str(row.get('unit', ''))
        if expected and actual and actual.lower() != expected.lower():
            unit_issues.append(f"   {row['metric_id']} ({row['year']}): expected '{expected}', got '{actual}'")

    if unit_issues:
        print(f"\n      ⚠️ Unit mismatches ({len(unit_issues)}) — may need conversion:")
        for issue in unit_issues[:10]:
            print(f"      {issue}")
        if len(unit_issues) > 10:
            print(f"      ... and {len(unit_issues) - 10} more")


print("✅ Export functions updated (7 sheets, unit mismatch detection)")


✅ Export functions updated (7 sheets, unit mismatch detection)


## Main Extraction Function

In [ ]:
# =============================================================================
# CELL 18: MAIN EXTRACTION PIPELINE — UPDATED COST CALC
# =============================================================================

def extract_company(company: str, input_dir: Path, output_dir: Path) -> Dict:
    """Run full extraction for one company."""
    print(f"\n{'='*70}")
    print(f"🔍 EXTRACTING: {company}")
    print(f"   Mode: {'FILTERED' if USE_FILTERED else 'UNFILTERED'}")
    print(f"   Model: {API_MODEL}")
    print(f"{'='*70}")

    company_input = input_dir / company

    if not company_input.exists():
        print(f"   ⚠️ No documents found: {company_input}")
        return {'company': company, 'error': 'No input'}

    # Extract from each source
    sust_results, sust_meta = extract_sustainability_pdfs(company, company_input)
    def14a_results, def14a_meta = extract_def14a_pdfs(company, company_input)
    tenk_results, tenk_meta = extract_10k_pdfs(company, company_input)

    # Combine
    pdf_results = sust_results
    sec_results = def14a_results + tenk_results

    # Add derived metrics
    pdf_derived = calculate_derived(pdf_results)
    sec_derived = calculate_derived(sec_results)
    pdf_all = pdf_results + pdf_derived
    sec_all = sec_results + sec_derived

    print(f"\n   📊 Results: {len(pdf_all)} PDF, {len(sec_all)} SEC")

    # Create comparison and fill template
    comparison = create_comparison(company, pdf_all, sec_all)
    template = create_template(company)
    final_df = fill_template(template, comparison)

    # Export
    export_results(final_df, comparison, company, output_dir)

    # Calculate costs using model-specific pricing
    total_input = (sust_meta.get('input_tokens', 0) +
                   def14a_meta.get('input_tokens', 0) +
                   tenk_meta.get('input_tokens', 0))
    total_output = (sust_meta.get('output_tokens', 0) +
                    def14a_meta.get('output_tokens', 0) +
                    tenk_meta.get('output_tokens', 0))

    pricing = MODEL_PRICING.get(API_MODEL, {"input": 1.00, "output": 5.00})
    input_cost = total_input / 1_000_000 * pricing["input"]
    output_cost = total_output / 1_000_000 * pricing["output"]

    print(f"\n   💰 Cost Breakdown ({API_MODEL}):")
    print(f"      Input:  {total_input:,} tokens (${input_cost:.3f})")
    print(f"      Output: {total_output:,} tokens (${output_cost:.3f})")
    print(f"      TOTAL:  ${input_cost + output_cost:.3f}")

    # Save extraction log
    extraction_log = {
        'company': company,
        'run_label': RUN_LABEL,
        'model': API_MODEL,
        'use_filtered': USE_FILTERED,
        'filter_version': FILTER_VERSION if USE_FILTERED else None,
        'input_dir': str(input_dir),
        'sustainability': {
            'files': len([f for f in company_input.glob("*Report_*.pdf")]),
            'values': len(sust_results),
            'input_tokens': sust_meta.get('input_tokens', 0),
            'output_tokens': sust_meta.get('output_tokens', 0),
        },
        'def14a': {
            'files': len([f for f in company_input.glob("*DEF_14A*.pdf")]),
            'values': len(def14a_results),
            'input_tokens': def14a_meta.get('input_tokens', 0),
            'output_tokens': def14a_meta.get('output_tokens', 0),
        },
        'tenk': {
            'files': len([f for f in company_input.glob("*10-K*.pdf")]),
            'values': len(tenk_results),
            'input_tokens': tenk_meta.get('input_tokens', 0),
            'output_tokens': tenk_meta.get('output_tokens', 0),
        },
        'total_input_tokens': total_input,
        'total_output_tokens': total_output,
        'total_cost': input_cost + output_cost,
        'coverage': round(final_df['is_filled'].mean() * 100, 1),
        'filled_metrics': int(final_df['is_filled'].sum()),
        'total_metrics': len(final_df),
    }

    log_path = LOG_DIR / f"{company}_extraction_log.json"
    with open(log_path, 'w') as f:
        json.dump(extraction_log, f, indent=2, default=str)

    # Also returning the raw extracted data allows re-export without re-running the llm Pipeline later
    extraction_log['_raw'] = {
        'pdf_all': pdf_all,
        'sec_all': sec_all,
    }

    return extraction_log


print("✅ Main extraction pipeline defined")

✅ Main extraction pipeline defined


## \# Run Extraction

In [ ]:
# =============================================================================
# CELL 19: RUN EXTRACTION
# =============================================================================

# TEST_COMPANIES = [
#     # "Advanced_Micro_Devices",
#     "Nvidia",
# ]

print(f"{'#'*70}")
print(f"# V10 LLM EXTRACTION PIPELINE")
print(f"{'#'*70}")
print(f"Model: {API_MODEL}")
print(f"Mode: {'FILTERED (v' + FILTER_VERSION + ')' if USE_FILTERED else 'UNFILTERED'}")
print(f"Run label: {RUN_LABEL}")
print(f"Input: {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Companies: {TEST_COMPANIES}")
print(f"{'#'*70}")

######################################################################
# V10 LLM EXTRACTION PIPELINE
######################################################################
Model: claude-haiku-4-5-20251001
Mode: FILTERED (v4)
Run label: filtered_v4
Input: /content/drive/MyDrive/ESG Data/filtered_documents_4
Output: /content/drive/MyDrive/ESG Data/extracted_data/filtered_v4
Companies: ['Dell_Technologies', 'HP_Inc']
######################################################################


In [ ]:
# =============================================================================
# CELL 20: PROCESS COMPANIES
# =============================================================================

all_results = []

for company in TEST_COMPANIES:
    result = extract_company(company, INPUT_DIR, OUTPUT_DIR)
    all_results.append(result)
    print(f"\n{'='*70}")


🔍 EXTRACTING: Dell_Technologies
   Mode: FILTERED
   Model: claude-haiku-4-5-20251001

   📄 Sustainability PDFs
      Found: 7 files
      📄 FILTERED_Report_Dell_Technologies_2024_3.pdf
         Size: 2.27 MB, 35 pages

      📊 Extraction Analysis for FILTERED_Report_Dell_Technologies_2024_3.pdf:
         Valid:          26
         Invalid year:   0
         Invalid metric: 0
         Missing value:  0
         ✅ 26 values (83,675 in / 1,710 out)
      📄 FILTERED_Report_Dell_Technologies_2023_15.pdf
         Size: 0.08 MB, 36 pages

      📊 Extraction Analysis for FILTERED_Report_Dell_Technologies_2023_15.pdf:
         Valid:          9
         Invalid year:   0
         Invalid metric: 0
         Missing value:  0
         ✅ 9 values (70,167 in / 816 out)
      📄 FILTERED_Report_Dell_Technologies_2023_90.pdf
         Size: 3.98 MB, 36 pages

      📊 Extraction Analysis for FILTERED_Report_Dell_Technologies_2023_90.pdf:
         Valid:          23
         Invalid year:   0
        

In [ ]:
# =============================================================================
# CELL 21: FINAL SUMMARY
# =============================================================================

print(f"\n{'='*70}")
print(f"V10 EXTRACTION COMPLETE")
print(f"{'='*70}")
print(f"Mode: {'FILTERED (v' + FILTER_VERSION + ')' if USE_FILTERED else 'UNFILTERED'}")
print(f"Model: {API_MODEL}")

pricing = MODEL_PRICING.get(API_MODEL, {"input": 1.00, "output": 5.00})
total_input = sum(r.get('total_input_tokens', 0) for r in all_results if 'error' not in r)
total_output = sum(r.get('total_output_tokens', 0) for r in all_results if 'error' not in r)
total_cost = sum(r.get('total_cost', 0) for r in all_results if 'error' not in r)
avg_coverage = sum(r.get('coverage', 0) for r in all_results if 'error' not in r)
valid_count = sum(1 for r in all_results if 'error' not in r)
avg_coverage = avg_coverage / valid_count if valid_count > 0 else 0

print(f"\nCompanies processed: {valid_count}")
print(f"Average coverage: {avg_coverage:.1f}%")

print(f"\n💰 TOTAL COST ({API_MODEL}):")
print(f"   Input tokens:  {total_input:,} (${total_input / 1_000_000 * pricing['input']:.2f})")
print(f"   Output tokens: {total_output:,} (${total_output / 1_000_000 * pricing['output']:.2f})")
print(f"   TOTAL:         ${total_cost:.2f}")

# Estimate for full 68-company run
if valid_count > 0:
    per_company_cost = total_cost / valid_count
    estimated_full = per_company_cost * 68
    print(f"\n📊 Estimated cost for 68 companies: ${estimated_full:.2f}")

print(f"\n📁 Output: {OUTPUT_DIR}")

print(f"\n📊 Per-Company Results:")
for r in all_results:
    if 'error' not in r:
        print(f"   {r['company']}: {r['coverage']:.1f}% coverage, "
              f"{r['filled_metrics']}/{r['total_metrics']} metrics, ${r['total_cost']:.3f}")
    else:
        print(f"   {r['company']}: ERROR - {r.get('error', 'Unknown')}")



V10 EXTRACTION COMPLETE
Mode: FILTERED (v4)
Model: claude-haiku-4-5-20251001

Companies processed: 2
Average coverage: 74.5%

💰 TOTAL COST (claude-haiku-4-5-20251001):
   Input tokens:  2,386,337 ($2.39)
   Output tokens: 37,796 ($0.19)
   TOTAL:         $2.58

📊 Estimated cost for 68 companies: $87.56

📁 Output: /content/drive/MyDrive/ESG Data/extracted_data/filtered_v4

📊 Per-Company Results:
   Dell_Technologies: 70.6% coverage, 180/255 metrics, $1.244
   HP_Inc: 78.4% coverage, 200/255 metrics, $1.332


In [ ]:
# Re-export for a specific company using data already in memory
company = "Advanced_Micro_Devices"
for r in all_results:
    if r.get('company') == company:
        comparison = create_comparison(company, r['_raw']['pdf_all'], r['_raw']['sec_all'])
        template = create_template(company)
        final_df = fill_template(template, comparison)
        export_results(final_df, comparison, company, OUTPUT_DIR)
        break

❌ Raw results no longer in memory — need to re-extract


### Testing And Debugging


In [ ]:
def retry_single_pdf(company: str, filename: str, input_dir: Path, doc_type: str = "sustainability_pdf"):
    """
    Retry extraction on a single PDF file.

    Args:
        company: Company name
        filename: PDF filename (e.g., "Report_Advanced_Micro_Devices_2023_94.pdf")
        input_dir: Input directory (INPUT_DIR)
        doc_type: "sustainability_pdf", "sec_def14a", or "sec_10k"

    Usage:
        results, meta = retry_single_pdf("Advanced_Micro_Devices",
                                          "Report_Advanced_Micro_Devices_2023_94.pdf", INPUT_DIR)
    """
    pdf_path = Path(input_dir) / company / filename

    if not pdf_path.exists():
        print(f"❌ File not found: {pdf_path}")
        return [], {}

    metrics = get_applicable_metrics(SECTOR)
    metrics_text = format_metrics_for_prompt(metrics)

    if doc_type == "sustainability_pdf":
        prompt = PROMPT_SUSTAINABILITY.format(
            company=company, metric_count=len(metrics), metrics=metrics_text)
    elif doc_type == "sec_def14a":
        fiscal_year = get_fiscal_year_def14a(filename)
        prompt = PROMPT_DEF14A.format(company=company, fiscal_year=fiscal_year)
    elif doc_type == "sec_10k":
        fiscal_year = get_fiscal_year_10k(filename, company)
        prompt = PROMPT_10K.format(company=company, fiscal_year=fiscal_year)
    else:
        print(f"❌ Unknown doc_type: {doc_type}")
        return [], {}

    results, meta = extract_from_pdf(str(pdf_path), company, prompt, doc_type)

    print(f"\n📊 Results: {len(results)} values extracted")
    for r in results[:5]:
        print(f"   {r.get('metric_id')}: {r.get('value')} {r.get('unit')} ({r.get('year')})")
    if len(results) > 5:
        print(f"   ... and {len(results) - 5} more")

    return results, meta

# Usage:
# results, meta = retry_single_pdf("Advanced_Micro_Devices", "Report_Advanced_Micro_Devices_2023_94.pdf", INPUT_DIR)

In [ ]:
results, meta = retry_single_pdf("Advanced_Micro_Devices", "Report_Advanced_Micro_Devices_2023_94.pdf", INPUT_DIR)

      📄 Report_Advanced_Micro_Devices_2023_94.pdf
         Size: 1.20 MB
         ❌ Error: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.0.content.0.pdf.source.base64.data: A maximum of 100 PDF pages may be provided.'}, 'request_id': 'req_011CZQ9YGtewMrba1UHUXq2T'}

📊 Results: 0 values extracted


In [ ]:
results, meta = retry_single_pdf("Advanced_Micro_Devices", "SEC_10-K_Advanced_Micro_Devices_2021.pdf", INPUT_DIR)

      📄 SEC_10-K_Advanced_Micro_Devices_2021.pdf
         Size: 0.64 MB
         ❌ Error: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'prompt is too long: 205777 tokens > 200000 maximum'}, 'request_id': 'req_011CZQ9dEE48zXAASDbM8ML9'}

📊 Results: 0 values extracted


In [ ]:
results, meta = retry_single_pdf("Advanced_Micro_Devices", "SEC_DEF_14A_Advanced_Micro_Devices_2021.pdf", INPUT_DIR)

      📄 SEC_DEF_14A_Advanced_Micro_Devices_2021.pdf
         Size: 6.14 MB
         ❌ Error: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'prompt is too long: 209916 tokens > 200000 maximum'}, 'request_id': 'req_011CZQ9jGY8ersAsQN6W8UCj'}

📊 Results: 0 values extracted
